This notebook is used to add and delete data to/from the Artifactory repository by course managers.
VITO artifactory credentials are required for this.

### Functions

In [13]:
import requests
from loguru import logger
import time
import os
from pathlib import Path


def _run_request(method: str, url: str, **kwargs) -> requests.Response:
    """Run an HTTP request with retries and return the response.
    Parameters
    ----------
    method : str
        HTTP method to be used
    url : str
        URL to send the request to
    kwargs : dict
        Additional keyword arguments, may include `retries`, `wait` and `logging_msg`
    Raises
    ------
    RuntimeError
        if the command fails after all retries
    Returns
    -------
    requests.Response
        The response of the http request
    """
    retries = kwargs.pop("retries", 3)
    wait = kwargs.pop("wait", 2)
    logging_msg = kwargs.pop("logging_msg", "Request")

    for attempt in range(retries):
        try:
            logger.debug(f"{logging_msg} (Attempt {attempt + 1})")
            response = requests.request(method, url, **kwargs)
            response.raise_for_status()
            logger.debug("Execution successful")
            return response
        except requests.RequestException as e:
            logger.warning(f"Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(wait)
            else:
                logger.error(f"Failed to execute request: {url}")
                raise
    raise RuntimeError(f"Failed to execute request: {url}")


def upload_file(srcpath, dstpath, username, password, retries=3, wait=2):
    """Upload a file to Artifactory.
    Parameters
    ----------
    srcpath : Path
        Path to csv file that needs to be uploaded to Artifactory.
    dstpath : str
        Full link to the target location in Artifactory.
    username : str
        Artifactory username.
    password : str
        Artifactory password.
    retries : int, optional
        Number of retries, by default 3
    wait : int, optional
        Seconds to wait in between retries, by default 2
    Returns
    -------
    str
        Full link to the target location in Artifactory.

    """
    url = dstpath
    with open(srcpath, "rb") as f:
        file_content = f.read()  # Read the file content as binary
        headers = {
            "Content-Type": "application/octet-stream",  # Set the appropriate content type
        }
        response = _run_request(
            "PUT",
            url,
            data=file_content,  # Send raw file content in the request body
            headers=headers,
            auth=(username, password),
            logging_msg=f"Uploading `{srcpath}` to `{dstpath}`",
            retries=retries,
            wait=wait,
        )
    return response.json()["downloadUri"]


def _get_artifactory_credentials():
    """Get credentials for upload and delete operations on Artifactory.
    Returns
    -------
    tuple (str, str)
        Tuple containing the Artifactory username and password.
    Raises
    ------
    ValueError
        if ARTIFACTORY_USERNAME or ARTIFACTORY_PASSWORD are not set as environment variables.
    """

    artifactory_username = os.getenv("ARTIFACTORY_USERNAME")
    artifactory_password = os.getenv("ARTIFACTORY_PASSWORD")

    if not artifactory_username or not artifactory_password:
        raise ValueError(
            "Artifactory credentials not found. "
            "Please set ARTIFACTORY_USERNAME and ARTIFACTORY_PASSWORD environment variables."
        )

    return artifactory_username, artifactory_password


def delete_file(srcpath: str, retries=3, wait=2):
    """Delete a file from Artifactory.
    Parameters
    ----------
    srcpath : str
        Path to the legend file in Artifactory.
    retries : int, optional
        Number of retries, by default 3
    wait : int, optional
        Seconds to wait in between retries, by default 2
    """
    # Get Artifactory credentials
    artifactory_username, artifactory_password = _get_artifactory_credentials()

    _run_request(
        "DELETE",
        srcpath,
        auth=(artifactory_username, artifactory_password),
        logging_msg=f"Deleting legend file: {srcpath}",
        retries=retries,
        wait=wait,
    )
    

def download_file(
    dstpath: Path,
    srcpath: str,
    retries=3,
    wait=2,
) -> Path:
    """Download a file from Artifactory.
    Parameters
    ----------
    dstpath : Path
        Folder where the legend needs to be downloaded to.
    srcpath : str
        Full path to the file in Artifactory.
    retries : int, optional
        Number of retries, by default 3
    wait : int, optional
        Seconds to wait in between retries, by default 2
    Returns
    -------
    Path
        Path to the downloaded file.
    """

    # Construct target path
    dstpath.mkdir(parents=True, exist_ok=True)
    filename = srcpath.split("/")[-1]
    download_file = dstpath / filename

    response = _run_request(
        "GET",
        srcpath,
        logging_msg=f"Downloading file: {filename}",
        retries=retries,
        wait=wait,
    )

    with open(download_file, "wb") as f:
        f.write(response.content)

    return download_file

### Upload a file

In [18]:
## UPLOAD A FILE

# === Authentication === #
artifactory_username, artifactory_password = _get_artifactory_credentials()

# === Configuration ===
artifactory_url = "https://artifactory.vgt.vito.be/artifactory"
repository = "auxdata-public"
subfolder = "vito_agri/tutorials/data"

# Local file to upload
file_paths = [Path("./parcel/data/S2_L2A_Malawi_10m_small.nc"),
              Path("./parcel/data/S2_L2A_Malawi_20m_small.nc"),
              Path("./parcel/data/S2_L2A_Malawi_10m.nc"),
              Path("./parcel/data/S2_L2A_Malawi_20m.nc"),
              Path("./parcel/data/Malawi_normalized_data_2023.gpkg"),]

for file_path in file_paths:
    # Final upload URL (automatically creates folders)
    filename = file_path.name
    upload_url = f"{artifactory_url}/{repository}/{subfolder}/{filename}"

    # === Upload file ===
    test = upload_file(
        srcpath=file_path,
        dstpath=upload_url,
        username=artifactory_username,
        password=artifactory_password,
    )


2025-04-14 13:59:26.387 | DEBUG    | __main__:_run_request:33 - Uploading `parcel/data/S2_L2A_Malawi_10m_small.nc` to `https://artifactory.vgt.vito.be/artifactory/auxdata-public/vito_agri/tutorials/data/S2_L2A_Malawi_10m_small.nc` (Attempt 1)
2025-04-14 13:59:27.016 | DEBUG    | __main__:_run_request:36 - Execution successful
2025-04-14 13:59:29.269 | DEBUG    | __main__:_run_request:33 - Uploading `parcel/data/S2_L2A_Malawi_20m_small.nc` to `https://artifactory.vgt.vito.be/artifactory/auxdata-public/vito_agri/tutorials/data/S2_L2A_Malawi_20m_small.nc` (Attempt 1)
2025-04-14 13:59:30.331 | DEBUG    | __main__:_run_request:36 - Execution successful
2025-04-14 13:59:30.488 | DEBUG    | __main__:_run_request:33 - Uploading `parcel/data/S2_L2A_Malawi_10m.nc` to `https://artifactory.vgt.vito.be/artifactory/auxdata-public/vito_agri/tutorials/data/S2_L2A_Malawi_10m.nc` (Attempt 1)
2025-04-14 13:59:40.189 | DEBUG    | __main__:_run_request:36 - Execution successful
2025-04-14 13:59:40.398 | DE

### Download a file

In [19]:
dst_path = Path("./parcel/data/downloaded")
download_file(dst_path, test)

2025-04-14 14:00:29.860 | DEBUG    | __main__:_run_request:33 - Downloading file: Malawi_normalized_data_2023.gpkg (Attempt 1)


2025-04-14 14:00:29.918 | DEBUG    | __main__:_run_request:36 - Execution successful


PosixPath('parcel/data/downloaded/Malawi_normalized_data_2023.gpkg')

### Delete a file

In [17]:
# input should be full path on artifactory
delete_file(test)

2025-04-14 13:59:20.679 | DEBUG    | __main__:_run_request:33 - Deleting legend file: https://artifactory.vgt.vito.be/artifactory/auxdata-public/vito_agri/tutorials/data/S2_L2A_Malawi_10m_subset.nc (Attempt 1)
2025-04-14 13:59:20.744 | DEBUG    | __main__:_run_request:36 - Execution successful
